In [2]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

In [3]:
from config import CATEGORY_INITIAL_RESULTS_PATH, CATEGORY_ROUND0_SUMMARY_PATH
from scoring_utils import compute_proxy_score

In [4]:
category_initial_results_df = pd.read_csv(CATEGORY_INITIAL_RESULTS_PATH)

print("Loaded generation results shape:", category_initial_results_df.shape)
print("Columns:", category_initial_results_df.columns.tolist())
display(category_initial_results_df.head(10))

Loaded generation results shape: (390, 11)
Columns: ['ape_round', 'parent_prompt_id', 'subcategory', 'prompt_id', 'prompt_family', 'prompt_variant', 'prompt_text', 'target_word', 'pos', 'reference_cue', 'generated_cue']


,ape_round,parent_prompt_id,subcategory,prompt_id,prompt_family,prompt_variant,prompt_text,target_word,pos,reference_cue,generated_cue
0,0,NaN,交通場所,交通場所_p01,category_meta,v1_meta_base,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通場所詞彙，並...,機場,名詞,內部有免稅商店和餐廳,NaN
1,0,NaN,交通場所,交通場所_p02,category_meta,v2_discriminative,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通場所詞彙，並...,機場,名詞,內部有免稅商店和餐廳,NaN
2,0,NaN,交通場所,交通場所_p03,category_meta,v3_life_context,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通場所詞彙，並...,機場,名詞,內部有免稅商店和餐廳,NaN
3,0,NaN,交通工具,交通工具_p01,category_meta,v1_meta_base,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通工具詞彙，並...,公車,名詞,票價便宜,需要刷卡進站
4,0,NaN,交通工具,交通工具_p01,category_meta,v1_meta_base,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通工具詞彙，並...,摩托車,名詞,需要加汽油或電力才能運行,需要開闢道路才能行駛
5,0,NaN,交通工具,交通工具_p01,category_meta,v1_meta_base,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通工具詞彙，並...,汽車,名詞,需要加油才能行駛,你需要在等人時看手機
6,0,NaN,交通工具,交通工具_p01,category_meta,v1_meta_base,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通工具詞彙，並...,火車,名詞,"需要購票搭乘,有分自由座和對號座",需要購票才能登車
7,0,NaN,交通工具,交通工具_p01,category_meta,v1_meta_base,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通工具詞彙，並...,自行車,名詞,運動或代步皆適合,騎行方便，速度快，適合短途出行
8,0,NaN,交通工具,交通工具_p01,category_meta,v1_meta_base,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通工具詞彙，並...,高鐵,名詞,"透過電力驅動,行駛於專用軌道上",需要刷卡才能進站
9,0,NaN,交通工具,交通工具_p02,category_meta,v2_discriminative,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通工具詞彙，並...,公車,名詞,票價便宜,上車時要留意刷卡機，按正確順序購買車票


In [5]:
# clean text columns so nothing breaks silently
for col in ["generated_cue", "reference_cue", "subcategory", "prompt_id", "prompt_family", "prompt_variant", "prompt_text", "parent_prompt_id"]:
    if col in category_initial_results_df.columns:
        category_initial_results_df[col] = category_initial_results_df[col].fillna("").astype(str).str.strip()

if "ape_round" in category_initial_results_df.columns:
    category_initial_results_df["ape_round"] = category_initial_results_df["ape_round"].fillna(0)

print("Empty generated_cue rows:", (category_initial_results_df["generated_cue"] == "").sum())
print("Non-empty generated_cue rows:", (category_initial_results_df["generated_cue"] != "").sum())

Empty generated_cue rows: 110
Non-empty generated_cue rows: 280


In [6]:
scored_rows = []

for _, row in category_initial_results_df.iterrows():
    generated_cue = row["generated_cue"]

    # skip only completely empty cues
    if generated_cue == "":
        continue

    score_dict = compute_proxy_score(
        generated_cue=generated_cue,
        target_word=row["target_word"],
        reference_cue=row["reference_cue"],
    )

    scored_rows.append({
        "ape_round": row["ape_round"],
        "parent_prompt_id": row["parent_prompt_id"],
        "subcategory": row["subcategory"],
        "prompt_id": row["prompt_id"],
        "prompt_family": row["prompt_family"],
        "prompt_variant": row["prompt_variant"],
        "prompt_text": row["prompt_text"],
        "target_word": row["target_word"],
        "pos": row["pos"],
        "reference_cue": row["reference_cue"],
        "generated_cue": row["generated_cue"],
        **score_dict,
    })

scored_df = pd.DataFrame(scored_rows)

print("Scored rows:", len(scored_df))
print("Scored shape:", scored_df.shape)
display(scored_df.head(10))

Scored rows: 280
Scored shape: (280, 15)


,ape_round,parent_prompt_id,subcategory,prompt_id,prompt_family,prompt_variant,prompt_text,target_word,pos,reference_cue,generated_cue,semantic_similarity,brevity_score,target_leak_penalty,proxy_score
0,0,,交通工具,交通工具_p01,category_meta,v1_meta_base,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通工具詞彙，並...,公車,名詞,票價便宜,需要刷卡進站,0.411199,1.0,0.0,0.587839
1,0,,交通工具,交通工具_p01,category_meta,v1_meta_base,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通工具詞彙，並...,摩托車,名詞,需要加汽油或電力才能運行,需要開闢道路才能行駛,0.462722,1.0,0.0,0.623905
2,0,,交通工具,交通工具_p01,category_meta,v1_meta_base,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通工具詞彙，並...,汽車,名詞,需要加油才能行駛,你需要在等人時看手機,0.133909,1.0,0.0,0.393736
3,0,,交通工具,交通工具_p01,category_meta,v1_meta_base,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通工具詞彙，並...,火車,名詞,"需要購票搭乘,有分自由座和對號座",需要購票才能登車,0.831019,1.0,0.0,0.881714
4,0,,交通工具,交通工具_p01,category_meta,v1_meta_base,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通工具詞彙，並...,自行車,名詞,運動或代步皆適合,騎行方便，速度快，適合短途出行,0.548727,1.0,0.0,0.684109
5,0,,交通工具,交通工具_p01,category_meta,v1_meta_base,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通工具詞彙，並...,高鐵,名詞,"透過電力驅動,行駛於專用軌道上",需要刷卡才能進站,0.302032,1.0,0.0,0.511423
6,0,,交通工具,交通工具_p02,category_meta,v2_discriminative,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通工具詞彙，並...,公車,名詞,票價便宜,上車時要留意刷卡機，按正確順序購買車票,0.379473,1.0,0.0,0.565631
7,0,,交通工具,交通工具_p02,category_meta,v2_discriminative,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通工具詞彙，並...,摩托車,名詞,需要加汽油或電力才能運行,需要駕駛員開著走,0.427704,1.0,0.0,0.599392
8,0,,交通工具,交通工具_p02,category_meta,v2_discriminative,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通工具詞彙，並...,汽車,名詞,需要加油才能行駛,需要車牌號碼才能啟動,0.421959,1.0,0.0,0.595372
9,0,,交通工具,交通工具_p02,category_meta,v2_discriminative,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通工具詞彙，並...,火車,名詞,"需要購票搭乘,有分自由座和對號座",需要提前預約座位，以免擠不進去,0.756312,1.0,0.0,0.829418


In [7]:
group_cols = [
    "ape_round",
    "parent_prompt_id",
    "subcategory",
    "prompt_id",
    "prompt_family",
    "prompt_variant",
    "prompt_text",
]

round0_summary = (
    scored_df.groupby(group_cols, as_index=False, dropna=False)
    .agg(
        mean_proxy_score=("proxy_score", "mean"),
        mean_similarity=("semantic_similarity", "mean"),
        mean_brevity=("brevity_score", "mean"),
        mean_target_leak=("target_leak_penalty", "mean"),
        n_examples=("generated_cue", "count"),
    )
    .sort_values(["subcategory", "mean_proxy_score"], ascending=[True, False])
    .reset_index(drop=True)
)

print("ROUND0 SUMMARY SHAPE:", round0_summary.shape)
print("ROUND0 SUMMARY COLUMNS:", round0_summary.columns.tolist())
display(round0_summary.head(20))

ROUND0 SUMMARY SHAPE: (55, 12)
ROUND0 SUMMARY COLUMNS: ['ape_round', 'parent_prompt_id', 'subcategory', 'prompt_id', 'prompt_family', 'prompt_variant', 'prompt_text', 'mean_proxy_score', 'mean_similarity', 'mean_brevity', 'mean_target_leak', 'n_examples']


,ape_round,parent_prompt_id,subcategory,prompt_id,prompt_family,prompt_variant,prompt_text,mean_proxy_score,mean_similarity,mean_brevity,mean_target_leak,n_examples
0,0,,交通工具,交通工具_p01,category_meta,v1_meta_base,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通工具詞彙，並...,0.613788,0.448268,1.000000,0.0,6
1,0,,交通工具,交通工具_p02,category_meta,v2_discriminative,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通工具詞彙，並...,0.595717,0.422453,1.000000,0.0,6
2,0,,交通工具,交通工具_p03,category_meta,v3_life_context,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通工具詞彙，並...,0.566154,0.380220,1.000000,0.0,5
3,0,,休閒娛樂,休閒娛樂_p01,category_meta,v1_meta_base,你是一位專門為高齡失語症患者設計語言治療提示句的專家。請根據目標詞彙最具辨識度的特徵，生成一...,0.686101,0.551573,1.000000,0.0,6
4,0,,休閒娛樂,休閒娛樂_p03,category_meta,v3_life_context,你是一位專門為高齡失語症患者設計語言治療提示句的專家。請根據目標詞彙最具辨識度的特徵，生成一...,0.667851,0.525501,1.000000,0.0,6
5,0,,休閒娛樂,休閒娛樂_p02,category_meta,v2_discriminative,你是一位專門為高齡失語症患者設計語言治療提示句的專家。請根據目標詞彙最具辨識度的特徵，生成一...,0.600724,0.429606,1.000000,0.0,5
6,0,,動作,動作_p03,category_meta,v3_life_context,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標動作詞彙，並生成...,0.548430,0.355828,0.997835,0.0,22
7,0,,動作,動作_p01,category_meta,v1_meta_base,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標動作詞彙，並生成...,0.534476,0.337547,0.993977,0.0,23
8,0,,動作,動作_p02,category_meta,v2_discriminative,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標動作詞彙，並生成...,0.522896,0.319395,0.997732,0.0,21
9,0,,動物,動物_p02,category_meta,v2_discriminative,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標動物詞彙，並生成...,0.697142,0.567346,1.000000,0.0,4


In [8]:
round0_summary.to_csv(CATEGORY_ROUND0_SUMMARY_PATH, index=False, encoding="utf-8-sig")
print("Saved:", CATEGORY_ROUND0_SUMMARY_PATH)

Saved: C:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Generator\data\interim\category_round0_prompt_summary.csv
